<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 6 — Machine Learning

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

**Tâche 1 — Régression :** prédire `duree_totale_h` (durée de traitement en heures)  
**Tâche 2 — Classification :** prédire `Cause.intervention` (type de sinistre)

**Référence :** [`docs/phase6_machine_learning.md`](../docs/phase6_machine_learning.md)  
**Données :** `data/dataset_complet.csv` — 98 935 × 34 colonnes


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, f1_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
dataset = pd.read_csv('data/dataset_complet.csv', encoding='utf-8')
print(f" Dataset : {dataset.shape[0]:,} lignes × {dataset.shape[1]} colonnes")

---
## Section 1 — Préparation des Données

### 1.1 Définition des features et de la cible

In [ ]:
# --- Features communes aux deux tâches
FEATURES_NUMERIQUES = [
    'agent_experience_j', 'agent_duree_travail_j', 'agent_temps_travail_pct',
    'delai_survenance_ouverture_j', 'nb_interventions', 'nb_agents_distincts',
    'mois_ouverture'
]

FEATURES_CATEGORIELLES = [
    'agent_lieu_travail', 'agent_contrat', 'agent_population', 'annee_ouverture'
]

TOUTES_FEATURES = FEATURES_NUMERIQUES + FEATURES_CATEGORIELLES

print("Features numériques  :", FEATURES_NUMERIQUES)
print("Features catégorielles:", FEATURES_CATEGORIELLES)
print(f"Total features       : {len(TOUTES_FEATURES)}")

In [ ]:
# --- Pipeline de prétraitement (M2 — évite le data leakage)
preprocesseur = ColumnTransformer(transformers=[
    ('numerique',    Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), FEATURES_NUMERIQUES),
    ('categoriel',   Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), FEATURES_CATEGORIELLES),
], remainder='drop')

print(" Pipeline de prétraitement défini")
print("   → Imputation médiane pour les numériques")
print("   → Imputation mode pour les catégoriels")
print("   → StandardScaler sur les numériques")
print("   → OneHotEncoder sur les catégoriels")

---
## Section 2 — Tâche 1 : Régression (`duree_totale_h`)

### Décision M3 : conserver les NaN via imputation (pas de dropna)
> Le `SimpleImputer` dans le pipeline gère les NaN — aucune observation n'est perdue.

In [ ]:
# --- Préparation données régression
df_reg = dataset[TOUTES_FEATURES + ['duree_totale_h']].copy()
df_reg = df_reg.dropna(subset=['duree_totale_h'])  # cible ne peut pas être NaN
df_reg['annee_ouverture'] = df_reg['annee_ouverture'].astype(str)

X_reg = df_reg[TOUTES_FEATURES]
y_reg = df_reg['duree_totale_h']

# --- Split train/test (80/20 — M1)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Observations pour régression : {len(df_reg):,}")
print(f"Train : {len(X_reg_train):,} | Test : {len(X_reg_test):,}")

In [ ]:
# --- Définition des modèles de régression
modeles_regression = {
    'LinearRegression'           : LinearRegression(),
    'RandomForestRegressor'      : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoostingRegressor'  : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultats_regression = []
modeles_entraines    = {}

for nom_modele, modele in modeles_regression.items():
    print(f"Entraînement : {nom_modele}...")
    
    pipeline_complet = Pipeline([
        ('preprocessing', preprocesseur),
        ('modele',        modele)
    ])
    
    # Entraînement
    pipeline_complet.fit(X_reg_train, y_reg_train)
    modeles_entraines[nom_modele] = pipeline_complet
    
    # Prédictions
    y_pred_test  = pipeline_complet.predict(X_reg_test)
    y_pred_train = pipeline_complet.predict(X_reg_train)
    
    # Métriques
    rmse_test  = np.sqrt(mean_squared_error(y_reg_test, y_pred_test))
    mae_test   = mean_absolute_error(y_reg_test, y_pred_test)
    r2_test    = r2_score(y_reg_test, y_pred_test)
    r2_train   = r2_score(y_reg_train, y_pred_train)
    
    resultats_regression.append({
        'Modèle'    : nom_modele,
        'RMSE test' : round(rmse_test, 3),
        'MAE test'  : round(mae_test, 3),
        'R² test'   : round(r2_test, 4),
        'R² train'  : round(r2_train, 4),
        'Surapprentissage': round(r2_train - r2_test, 4)
    })
    print(f"  RMSE={rmse_test:.3f} | MAE={mae_test:.3f} | R²={r2_test:.4f}")
    print()

df_resultats_reg = pd.DataFrame(resultats_regression).sort_values('R² test', ascending=False)
print("=== COMPARAISON DES MODÈLES — RÉGRESSION ===")
df_resultats_reg

In [ ]:
# --- Visualisation de la comparaison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Tâche 1 : Comparaison des modèles de Régression", fontsize=13, fontweight='bold')

noms    = df_resultats_reg['Modèle'].str.replace('Regressor','').str.replace('Regression','')
couleurs = ['#4472c4', '#ed7d31', '#70ad47']

axes[0].bar(noms, df_resultats_reg['RMSE test'], color=couleurs, edgecolor='white')
axes[0].set_title("RMSE (test) — plus bas = mieux")
axes[0].set_ylabel("RMSE (heures)")
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(noms, df_resultats_reg['R² test'], color=couleurs, edgecolor='white')
axes[1].set_title("R² (test) — plus haut = mieux")
axes[1].set_ylabel("R²")
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(noms, df_resultats_reg['Surapprentissage'], color=['#c00000' if v > 0.1 else '#70ad47'
            for v in df_resultats_reg['Surapprentissage']], edgecolor='white')
axes[2].set_title("R²train - R²test (surapprentissage)")
axes[2].set_ylabel("Différence")
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('data/phase6_regression_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

meilleur_nom_reg = df_resultats_reg.iloc[0]['Modèle']
print(f" Meilleur modèle de régression : {meilleur_nom_reg}")

---
## Section 3 — Importance des Variables (Régression)

### Analyse du meilleur modèle

In [ ]:
# --- Feature Importance du meilleur modèle
pipeline_meilleur_reg = modeles_entraines[meilleur_nom_reg]
modele_interne = pipeline_meilleur_reg.named_steps['modele']

# Récupérer les noms des features après OHE
noms_numeriques  = FEATURES_NUMERIQUES
ohe              = pipeline_meilleur_reg.named_steps['preprocessing'].transformers_[1][1].named_steps['ohe']
noms_ohe         = ohe.get_feature_names_out(FEATURES_CATEGORIELLES).tolist()
noms_toutes_features = noms_numeriques + noms_ohe

if hasattr(modele_interne, 'feature_importances_'):
    importances = modele_interne.feature_importances_
    df_importance = pd.DataFrame({
        'Feature'   : noms_toutes_features,
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(df_importance['Feature'][::-1], df_importance['Importance'][::-1],
            color='#4472c4', edgecolor='white')
    ax.set_title(f"Top 15 Features — {meilleur_nom_reg}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig('data/phase6_feature_importance_regression.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(df_importance.to_string(index=False))
else:
    print("LinearRegression — coefficients utilisés à la place des importances")
    coefs = np.abs(modele_interne.coef_)
    df_importance = pd.DataFrame({'Feature': noms_toutes_features, 'Coefficient absolu': coefs})
    print(df_importance.sort_values('Coefficient absolu', ascending=False).head(10).to_string(index=False))

---
## Section 4 — Tâche 2 : Classification (`Cause.intervention`)

### 4.1 Analyse et préparation de la cible

In [ ]:
# --- Distribution de la cible
df_clf = dataset[TOUTES_FEATURES + ['Cause.intervention']].dropna(subset=['Cause.intervention']).copy()
df_clf['annee_ouverture'] = df_clf['annee_ouverture'].astype(str)

distribution_causes = df_clf['Cause.intervention'].value_counts()
print(f"Observations disponibles : {len(df_clf):,}")
print(f"Modalités de Cause.intervention :")
df_dist = distribution_causes.reset_index()
df_dist.columns = ['Cause', 'N']
df_dist['%'] = (df_dist['N'] / len(df_clf) * 100).round(2)
print(df_dist.to_string(index=False))

In [ ]:
# --- Regroupement des modalités rares (< 2%) en 'Autres'
SEUIL_MODALITE_RARE_PCT = 2.0
seuil_absolu = int(len(df_clf) * SEUIL_MODALITE_RARE_PCT / 100)

modalites_frequentes = distribution_causes[distribution_causes >= seuil_absolu].index.tolist()
df_clf['Cause.intervention.regroupee'] = df_clf['Cause.intervention'].apply(
    lambda x: x if x in modalites_frequentes else 'Autres'
)

print(f"Modalités conservées (>= {seuil_absolu:,} obs) : {len(modalites_frequentes)}")
print(f"Modalités regroupées en 'Autres' : {len(distribution_causes) - len(modalites_frequentes)}")
print()
print("Distribution finale :")
print(df_clf['Cause.intervention.regroupee'].value_counts().to_string())

In [ ]:
# --- Encodage de la cible et split
encodeur_cible = LabelEncoder()
X_clf = df_clf[TOUTES_FEATURES]
y_clf = encodeur_cible.fit_transform(df_clf['Cause.intervention.regroupee'])

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"Observations pour classification : {len(df_clf):,}")
print(f"Train : {len(X_clf_train):,} | Test : {len(X_clf_test):,}")
print(f"Classes : {encodeur_cible.classes_.tolist()}")